# Упрощение текста
***
**Анастасия Добрынина**

Дана научная статья (можно брать юридические документы или что-то пододбное), нужно переписать её так, чтобы было понятно школьнику 10–12 лет.

Задача двойная: seq2seq модель генерирует упрощённый текст, эмбеддинги оценивают, насколько смысл сохранился

* Собрать датасет —  пары «научный текст / упрощённый референс» по 2–3 темам (например, биология + история). Параллельные пары можно генерировать через LLM как псевдо-таргет.
* Реализовать baseline — промптинг через HuggingFace Inference API: «перепиши этот текст так, чтобы было понятно школьнику». Без дообучения, просто zero-shot.
* Дообучить mT5-small или ruT5 на собранных парах через LoRA — сравнить с zero-shot baseline.
* Оценить качество двумя способами: автоматически: ROUGE vs референс + косинусное сходство эмбеддингов оригинала и упрощённого текста
* Проанализировать ошибки


In [1]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
# Установка необходимых библиотек
!pip install -q transformers peft datasets evaluate sentence-transformers
!pip install -q rouge_score pandas matplotlib tqdm
!pip install -q huggingface_hub
!pip install -q PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 18.8 MB/s eta 0:00:00


In [3]:
import os
import re
import time
import numpy as np
import fitz  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import Dataset
from huggingface_hub import InferenceClient
import torch
torch.manual_seed(42)

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate
from sentence_transformers import SentenceTransformer, util

## Датасет
Возьмем 4 лингвистические статьи на разные темы на английском длиной 20-40 страниц в формате PDF.

### Предобработка
Прямое извлечение текста из PDF часто сталкивается с проблемами, такими как склеенные слова, лишние переносы строк, колонтитулы, номера страниц и прочий "мусор". Чтобы получить чистый и пригодный для анализа текст, мы проведем предобработку в несколько этапов:

1.  **Первичная очистка с помощью регулярных выражений:** На этом шаге мы будем удалять типовые элементы, такие как URL-адреса, даты скачивания, строки, состоящие из заглавных букв (которые часто являются колонтитулами), и множественные пробелы. Также будут склеены разорванные слова.

2.  **Интеллектуальное извлечение блоков текста с помощью `PyMuPDF` (библиотеки `fitz`):** Вместо простого извлечения всего текста со страницы, `fitz` позволяет получать текст в виде структурных блоков. Это помогает лучше сегментировать абзацы и избегать проблем с неправильным порядком текста, характерным для PDF-файлов. Мы будем использовать функцию `page.get_text("blocks")`, которая возвращает кортежи с информацией о каждом блоке текста, включая сам текст. Это позволяет более точно идентифицировать и извлекать осмысленные абзацы.

3.  **Фильтрация и постобработка извлеченных абзацев:**
    *   **Удаление библиографии:** Мы добавим фильтры для исключения блоков, которые выглядят как библиографические записи (например, по количеству точек или ключевым словам, таким как "University Press").
    *   **Фильтрация по длине:** Чтобы получить абзацы, подходящие для суммаризации, мы будем отбирать блоки текста с длиной от 150 до 1000 символов. Слишком короткие блоки часто не несут достаточного контекста, а слишком длинные могут быть слишком сложными для LLM на этапе генерации.
    *   **Удаление дубликатов:** После извлечения всех абзацев мы удалим возможные дубликаты, чтобы каждый уникальный абзац был представлен только один раз.

Этот процесс позволит нам сформировать чистый датасет `complex_text`, который будет использоваться для дальнейшей работы с моделями упрощения.

In [4]:
FOLDER_PATH = '/content/drive/MyDrive/AppliedNN_papers'

In [5]:
def clean_block(text):
    """Очищает отдельный смысловой блок от мусора и переносов."""
    # Убираем URL и штампы скачивания
    text = re.sub(r'Downloaded from http\S+.*?\d{4}', '', text)
    # Убираем строки из ЗАГЛАВНЫХ букв (колонтитулы)
    text = re.sub(r'^[A-Z\s0-9]{15,}$', '', text, flags=re.MULTILINE)

    # Склеиваем переносы слов
    text = re.sub(r'-\n+', '', text)
    # Склеиваем строки в один нормальный абзац
    text = text.replace('\n', ' ')

    # Чистим ссылки
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\d{4}.*?\)', '', text)

    # Убираем множественные пробелы
    text = re.sub(r'[ \t]+', ' ', text).strip()
    return text



In [6]:
all_paragraphs = []

pdf_files = [f for f in os.listdir(FOLDER_PATH) if f.lower().endswith('.pdf')]

for filename in tqdm(pdf_files, desc="Умная обработка блоков"):
    filepath = os.path.join(FOLDER_PATH, filename)

    try:
        doc = fitz.open(filepath)

        for page in doc:
            # get_text("blocks") возвращает список кортежей.
            # 5-й элемент кортежа (индекс 4) — это текст физического абзаца.
            blocks = page.get_text("blocks")

            for b in blocks:
                block_text = b[4]
                cleaned_p = clean_block(block_text)

                # Жесткий фильтр на библиографию (если слишком много точек/тире)
                if cleaned_p.count('.') > 6 or "University Press" in cleaned_p:
                    continue

                # Фильтруем по длине
                if 150 < len(cleaned_p) < 1000:
                    all_paragraphs.append(cleaned_p)

        doc.close()

    except Exception as e:
        print(f"Ошибка при обработке {filename}: {e}")

# Убираем дубликаты
all_paragraphs = list(set(all_paragraphs))

print(f"Собрано чистых абзацев: {len(all_paragraphs)}")

Умная обработка блоков:   0%|          | 0/4 [00:00<?, ?it/s]

Собрано чистых абзацев: 290


In [8]:
# Упаковываем в pandas
df_dataset = pd.DataFrame({"complex_text": all_paragraphs})

# Перемешиваем датасет на всякий случай
df_dataset = df_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# первые 5 результатов
for i in range(5):
    print(f"======")
    print(df_dataset['complex_text'].iloc[i])


‘Who slept on the ﬂoor? CT ﬂoor-SUP P´eterF Peter aludt, slept ´es and (lehet, perhaps hogy) that sehol nowhere m´ashol other place nem not is too aludt slept senki nobody m´as. diﬀerent ‘It was Peter who slept ont he ﬂoor, but (it is possible that) nobody slept anywhere else.
Indefinite-first Inverse scope Results from Results from OVS (YES quantificational quantificational reading response) reading of odin  of dva  covert QR of covert QR of subject subject (costly) (costly)
The same can be observed e.g. in Hungarian : (22) only has the ‘not. . . everybody’ reading, although quantiﬁcational elements in Hungarian usually have a preference for surface scope:
One might argue that in the case of odin indefinites, the rate of YES responses . In an experimental study of scope in Mandarin and English with a paradigm similar to ours found a 0% acceptance rate of Mandarin equivalents of sentences such as One shark attacked every pirate in inverse scope scenarios. In contrast, in the English 

In [9]:
# Считаем длину в словах и символах
df_dataset['word_count'] = df_dataset['complex_text'].apply(lambda x: len(x.split()))
df_dataset['char_count'] = df_dataset['complex_text'].apply(len)

print("=== Ключевые метрики датасета ===")
print(f"Всего чистых абзацев: {len(df_dataset)}")
print(f"Средний объем абзаца (в словах): {df_dataset['word_count'].mean():.1f}")
print(f"Средний объем абзаца (в символах): {df_dataset['char_count'].mean():.1f}")
print(f"90% абзацев короче, чем: {df_dataset['char_count'].quantile(0.9):.0f} символов")

=== Ключевые метрики датасета ===
Всего чистых абзацев: 290
Средний объем абзаца (в словах): 67.0
Средний объем абзаца (в символах): 405.1
90% абзацев короче, чем: 732 символов


#### Оценка предобработки текста

Проведенная предобработка текста из PDF-документов показала хорошие результаты, позволив собрать достаточно чистый и пригодный для дальнейшего анализа датасет:

*   **Количество абзацев:** Было успешно собрано **290** уникальных абзацев, что является хорошей базой для обучения модели упрощения.

*   **Чистота текста:** Применение регулярных выражений и интеллектуальное извлечение блоков с помощью `PyMuPDF` (`fitz`) позволило эффективно удалить "мусор" (URL, колонтитулы, номера страниц, библиографические ссылки) и корректно обработать переносы строк, сделав текст читабельным и логически связным.

*   **Длина абзацев:** Фильтрация по длине (150-1000 символов) обеспечила наличие абзацев оптимального размера для задач суммаризации

*   **Удаление дубликатов:** Применение функции `set()` к собранным абзацам гарантировало отсутствие повторяющихся фрагментов, что улучшает качество и разнообразие обучающего датасета.

### Гнерация псевдо-таргетов

Поскольку наши исходные тексты извлечены из англоязычных научных статей, мы будем обучать модель упрощать тексты на оригинальном языке.

В этой ячейке мы используем подход **zero-shot prompting**. Мы обращаемся к большой языковой модели (LLM) через Hugging Face API и передаем ей системный промпт на английском языке. Роль модели — школьный учитель, который читает сложный академический абзац и переписывает его так, чтобы суть была понятна 10–12-летнему ребенку. На выходе мы получаем синтетический параллельный корпус `[complex_english] -> [simplified_english]`, который станет основой для дообучения нашей компактной seq2seq модели на следующем этапе.

Модель `Mixtral-8x7B-Instruct` выбрана благодаря её высокой способности точно следовать многоступенчатым инструкциям в режиме zero-shot и эффективно переписывать плотный академический текст на простой английский без потери исходного смысла.

In [10]:
from google.colab import userdata

# Безопасно достаем токен из хранилища Colab
HF_TOKEN = userdata.get('Colab_API')

In [11]:
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
client = InferenceClient(model=MODEL_ID, token=HF_TOKEN)


In [11]:
def safe_generate(text, max_retries=4):
    """
    Сбои API могут привести к потере большого количества пар
    Безопасная функция генерации делает несколько попыток при сбоях API.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert middle school teacher. Rewrite the complex academic text "
                "so that a 10-12 year old student can easily understand it.\n"
                "Rules:\n1. Use simple vocabulary and short sentences.\n"
                "2. Avoid complex syntactic structures.\n"
                "3. Output ONLY the simplified text. Do not add introductory phrases."
            )
        },
        {"role": "user", "content": f"Original academic text:\n{text}"}
    ]

    for attempt in range(max_retries):
        try:
            response = client.chat_completion(
                messages=messages,
                max_tokens=300,
                temperature=0.2
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg:
                # Если сервер перегружен, ждем 10 секунд и пробуем снова
                time.sleep(10)
            else:
                # При других ошибках (например, сбой сети) ждем 5 секунд
                time.sleep(5)

    # Если все попытки исчерпаны, возвращаем None (бывает крайне редко)
    return None

In [14]:
# 2. Подготовка к умному циклу
results = []
# Файл, куда будет записываться прогресс
checkpoint_file = "/content/drive/MyDrive/AppliedNN_papers/dataset_backup.csv"


# 3. Запускаем цикл с прогресс-баром
for index, row in tqdm(df_dataset.iterrows(), total=len(df_dataset)):
    complex_text = row['complex_text']

    # Генерируем перевод
    simplified_text = safe_generate(complex_text)
    results.append(simplified_text)

    # Каждые 10 шагов сохраняем резервную копию на диск
    if (index + 1) % 10 == 0:
        temp_df = df_dataset.iloc[:index+1].copy()
        temp_df['simplified_text'] = results
        temp_df.to_csv(checkpoint_file, index=False)

    # Обязательная пауза, чтобы не получить бан от Hugging Face
    time.sleep(1.5)

# 4. Финальная сборка датасета
df_dataset['simplified_text'] = results

# Смотрим, сколько данных удалось спасти
total_before = len(df_dataset)
df_dataset = df_dataset.dropna(subset=['simplified_text']).reset_index(drop=True)
total_after = len(df_dataset)

# Сохраняем финальный идеальный датасет
df_dataset.to_csv("/content/drive/MyDrive/AppliedNN_papers/final_dataset_complete.csv", index=False)


print(f"Финальный датасет сохранен в /content/drive/MyDrive/AppliedNN_papers/final_dataset_complete.csv")

In [12]:
# ИЛИ открываем предсохарненный файл с псевдо-таргетом, чтобы не генерировать при повторном запуске
file_path = '/content/drive/MyDrive/AppliedNN_papers/final_dataset_complete.csv'

df_dataset = pd.read_csv(file_path)
print(f"Всего готовых пар: {len(df_dataset)}")

# 4. Выводим первые пару строк для проверки
display(df_dataset.head(2))

Всего готовых пар: 288


,complex_text,word_count,char_count,simplified_text
0,The fact that contrastive focus leads to avail...,66,414,Here's the simplified text:\n\nWhen we put the...
1,"Here, we adopt the noncartographic approach. F...",123,732,Here's the simplified text:\n\nWe're going to ...


In [15]:
def remove_llm_chatter(text):
    """
    Удаляет типичные вежливые вступления чат-ботов из начала текста.
    """
    # Паттерн ищет типичные фразы до двоеточия и переноса строки
    # Например: "Here is the rewritten text:\n\n" или "Simplified text:"
    pattern = r"^(Here is|Here's|Sure|Here is the|Here's the|Let's).*?:\s*\n*"

    # Заменяем найденное вступление на пустоту
    clean_text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()

    # Дополнительно убираем просто "Simplified text:" если попалось
    clean_text = re.sub(r"^Simplified text:\s*\n*", "", clean_text, flags=re.IGNORECASE).strip()

    return clean_text


In [16]:
# Применяем очистку ко всему датасету
df_dataset['simplified_text'] = df_dataset['simplified_text'].apply(remove_llm_chatter)

# И обновляем Train/Test выборки, чтобы они тоже очистились
train_df, test_df = train_test_split(df_dataset, test_size=0.25, random_state=42)

In [17]:
# Проверка на чистоту
for i in range(10):
  print("------")
  print(train_df['simplified_text'].iloc[i][:50] + "...")

------
When someone says "I play" with a head shake, it's...
------
When the speaker starts talking about how many lun...
------
We think that people don't agree on this because o...
------
Speakers sometimes use special words to show how m...
------
The subject is in a special place in the sentence....
------
3.5.3 Interim Summary: Making Sentences

We found ...
------
When someone says "no", they usually shake their h...
------
The 38th Meeting of the Chicago Linguistic Society...
------
A person is giving a tour of an old town in Northa...
------
Linguistic theory is important for understanding h...


In [18]:
print("ОРИГИНАЛ (Complex):")
print(train_df['complex_text'].iloc[20])
print("\nУПРОЩЕННЫЙ РЕФЕРЕНС (Target):")
print(train_df['simplified_text'].iloc[20])

ОРИГИНАЛ (Complex):
3.1.2 Picture Types Each sentence type was presented with two distinct picture types, control and test. The control pictures made the sentence true on both surface and inverse scope. For example, for the sentence type in (15a), the control picture showed one specific doctor examining all three patients (while two other doctors stood by and did nothing): this picture makes (15a) true both on surface scope (there is one doctor who examined all the patients) and on inverse scope (for every patient, one doctor examined him/her). For reasons of space, we do not report the results with control pictures here; performance with control pictures was close to ceiling, which indicates that participants were paying attention.

УПРОЩЕННЫЙ РЕФЕРЕНС (Target):
There are two kinds of pictures for each sentence. One kind of picture makes the sentence true in two ways. For example, for a sentence like "A doctor is examining three patients," a picture that shows one doctor examining all

### Шаг 2: Zero-shot Baseline (Базовое решение без дообучения)

Перед тем как применять LoRA и дообучать модель на нашем собранном датасете, необходимо зафиксировать «базовую линию» (baseline). Мы должны понять, как модель справляется с задачей упрощения текста «из коробки», не видя наших примеров.

В качестве архитектуры мы будем использовать `seq2seq` модель `google/flan-t5-base`. Мы выбрали эту небольшую модель намеренно: она значительно уступает в «интеллекте» использованной для генерации псевдо-таргетов Llama-3-8B, поэтому её слабые результаты в режиме zero-shot станут идеальной отправной точкой для демонстрации эффективности нашего обучения

Мы подадим ей тексты из нашей отложенной тестовой выборки (`test_df`) с простой инструкцией переписать текст для школьника. Результаты мы сохраним в отдельную колонку `baseline_text`. На Шаге 4 мы сравним эти сырые результаты с ответами дообученной модели, используя метрики ROUGE и косинусное сходство эмбеддингов.

In [19]:
# Загрузка результатов Baseline с Диска (при перезапуске)
baseline_path = '/content/drive/MyDrive/AppliedNN_papers/baseline_results.csv'
baseline_df = pd.read_csv(baseline_path)

print(f"✅ Бейзлайны успешно загружены! Всего строк: {len(baseline_df)}")
display(baseline_df.head(2))

✅ Бейзлайны успешно загружены! Всего строк: 72


,complex_text,word_count,char_count,simplified_text,baseline_text
0,"Krivoshein De Canese, Natalia, Carlos Martinez...",21,175,Tetagua Remimombe'u: Paraguayan Folk Tales. Th...,"Krivoshein De Canese, Natalia, Carlos Martinez..."
1,*This work was funded in part by a grant from...,48,308,This project was helped by a grant from the Eu...,* This work was funded in part by a grant from...


In [20]:

# Используем FLAN-T5, так как она лучше понимает инструкции в zero-shot режиме для seq2seq задач
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Проверяем наличие видеокарты (GPU) в Colab для ускорения работы
device = "cuda" if torch.cuda.is_available() else "cpu"

# Загружаем саму модель и отправляем ее на нужное устройство
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print(f"Модель загружена на {device.upper()}")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Модель загружена на CPU


In [18]:
# ИЛИ обучение модели вместо загрузки

def generate_baseline(text):
    """
    Генерирует упрощение текста с помощью T5 без дообучения (zero-shot).
    """
    # Простой промпт для базовой модели
    prompt = f"Rewrite this scientific text so a 10-12 year old student can understand it easily:\n\n{text}"

    # Токенизируем входной текст
    # max_length=512, так как мы ранее убедились, что наши абзацы обычно не превышают этот лимит
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)

    # Генерируем ответ
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=250,
            num_beams=4, # Используем beam search для более связного текста
            early_stopping=True
        )

    # Декодируем токены обратно в читаемый текст
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [19]:
#Генерация baseline-ответов для тестовой выборки

# Создаем копию test_df, чтобы безопасно работать с данными
baseline_df = test_df.copy()

# Применяем функцию генерации только к тестовой выборке
tqdm.pandas(desc="Zero-shot Baseline")
baseline_df['baseline_text'] = baseline_df['complex_text'].progress_apply(generate_baseline)

Zero-shot Baseline:   0%|          | 0/72 [00:00<?, ?it/s]

In [21]:
# Сохранение результатов Baseline на Диск
baseline_path = '/content/drive/MyDrive/AppliedNN_papers/baseline_results.csv'
baseline_df.to_csv(baseline_path, index=False)

In [21]:
print("\n=== СРАВНЕНИЕ BASELINE И ПСЕВДО-ТАРГЕТА ===")
print("1. ОРИГИНАЛ (Complex text):")
print(baseline_df['complex_text'].iloc[20])
print("\n2. НАШ ИДЕАЛ (Pseudo-target):")
print(baseline_df['simplified_text'].iloc[20])
print("\n3. ZERO-SHOT БЕЗ ОБУЧЕНИЯ (FLAN-T5 Baseline):")
print(baseline_df['baseline_text'].iloc[20])


=== СРАВНЕНИЕ BASELINE И ПСЕВДО-ТАРГЕТА ===
1. ОРИГИНАЛ (Complex text):
H: (registering names and addresses) Leans toward child: Do you live in Barnwell? Child: <++> A: (child’s aunt): No, its her cousin that lives in Barnwell

2. НАШ ИДЕАЛ (Pseudo-target):
I'm writing down names and addresses. I'm leaning in to talk to a child. Do you live in Barnwell? The child says no. It's actually their cousin who lives in Barnwell.

3. ZERO-SHOT БЕЗ ОБУЧЕНИЯ (FLAN-T5 Baseline):
Barnwell is a small town in Yorkshire, England.


FLAN-T5 просто скопировала почти весь исходный текст. Она не поняла задачу упрощения, а просто решила «выбрать самый важный кусок». Это идеальный baseline — он наглядно показывает, что модель «не в теме».

## Шаг 3: Fine-tuning через LoRA (Адаптация модели)

Теперь мы проведем дообучение `flan-t5-base` на нашем синтетическом датасете, созданном с помощью Llama-3-8B. Чтобы сделать это быстро и эффективно на мощностях Google Colab, мы будем использовать метод **LoRA (Low-Rank Adaptation)**.

Суть метода заключается в «заморозке» основных весов модели и внедрении в её слои обучаемых матриц низкого ранга. Это позволяет:
1. **Экономить память:** Обучать всего 0.1%–1% от общего числа параметров.
2. **Предотвратить переобучение:** Маленькая модель не «зазубрит» данные, а адаптируется к стилю упрощения текста.
3. **Сохранить знания:** Модель не забудет правила английской грамматики, которые она выучила при претрейнинге.

#### Загрузка сохраненных весов модели (если нет ГПУ, чтобы обучить заново)

In [22]:
# Путь к сохраненным весам LoRA на Google Диске
lora_weights_path = "/content/drive/MyDrive/AppliedNN_papers/flan-t5-lora-weights"
model_name = "google/flan-t5-base"

# print("Загрузка токенизатора и базовой модели...")
# tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Мтод PEFT, который накладывает сохраненные адаптеры на чистую базовую модель
model = PeftModel.from_pretrained(base_model, lora_weights_path)

# Определяем устройство (GPU, если доступно, иначе CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Переводим модель в режим инференса
model.eval()

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
               

In [30]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,              # Ранг матриц (чем выше, тем больше обучаемых параметров)
    lora_alpha=32,    # Коэффициент масштабирования
    lora_dropout=0.1  # Регуляризация для предотвращения переобучения
)

# Создаем модель с адаптерами
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [31]:
# Конвертируем наш DataFrame в формат Hugging Face

train_dataset = Dataset.from_pandas(train_df[['complex_text', 'simplified_text']])

def preprocess_function(examples):
    inputs = ["rewrite for a 10-12 year old: " + doc for doc in examples["complex_text"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Токенизируем таргеты
    labels = tokenizer(text_target=examples["simplified_text"], max_length=250, truncation=True, padding="max_length")

    # ВАЖНОЕ ИСПРАВЛЕНИЕ: Заменяем токены паддинга на -100
    # PyTorch автоматически игнорирует метки -100 при расчете loss, так модель не будет учиться предсказывать пустоту
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = train_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/216 [00:00<?, ? examples/s]

In [32]:
# Data collator динамически дополняет (padding) батчи до одинаковой длины
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Настраиваем аргументы тренировки
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-lora-simplifier",
    learning_rate=1e-3,             # Для LoRA нужен rate выше, чем для обычного fine-tuning
    per_device_train_batch_size=4,  # Оптимально для бесплатной T4 GPU в Colab
    weight_decay=0.01,              # Регуляризация
    save_total_limit=2,             # Хранить только 2 последних чекпоинта, чтобы не забить диск
    num_train_epochs=5,             # 5-8 эпох обычно достаточно для маленького датасета
    predict_with_generate=True,
    fp16=False,                     # Отключаем fp16 для T5
    logging_steps=10,
    report_to="none"                # Отключаем интеграцию со сторонними логгерами вроде wandb
)

# Создаем объект Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    processing_class=tokenizer,
    data_collator=data_collator,
)

# Запускаем тренировку
trainer.train()

=========== Запуск обучения ===========


Step,Training Loss
10,2.132429
20,2.025588
30,1.999883
40,1.874861
50,2.009672
60,1.826917
70,1.754402
80,1.595574
90,1.684939
100,1.795587


TrainOutput(global_step=270, training_loss=1.6835381507873535, metrics={'train_runtime': 186.1369, 'train_samples_per_second': 5.802, 'train_steps_per_second': 1.451, 'total_flos': 742473282355200.0, 'train_loss': 1.6835381507873535, 'epoch': 5.0})

In [26]:
# Сохраняем веса обученных LoRA-адаптеров
model.save_pretrained("/content/drive/MyDrive/AppliedNN_papers/flan-t5-lora-weights")

Модель успешно дообучилась
* **Training Loss:** модель уловила паттерны упрощения текста, но избежала переобучения
* **Скорость:** Благодаря параметрически эффективному обучению (LoRA), процесс занял всего около 3 минут на бесплатном GPU.

## Шаг 4: Инференс (Генерация) и Сравнение результатов

Маленькая T5-base оснащена LoRA-адаптерами, которые усвоили стиль упрощения текста.

Чтобы проверить, сработала ли дистилляция знаний, мы прогоним через дообученную модель нашу отложенную тестовую выборку. Затем мы сведем все результаты в одну таблицу, чтобы наглядно увидеть эволюцию:
1. Исходный сложный текст.
2. Эталонный перевод от большой модели (Llama-3-8B).
3. Результат T5 «До» обучения (Zero-shot Baseline).
4. Результат T5 «После» обучения (LoRA Fine-tuned).

In [23]:
def generate_with_lora(text):
    """
    Генерирует текст с помощью нашей дообученной LoRA-модели.
    """
    # Важно: используем ровно тот же промпт, на котором учили!
    prompt = "rewrite for a 10-12 year old: " + text

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=250,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [24]:
# Добавляем новую колонку прямо в наш baseline_df для удобного сравнения
tqdm.pandas(desc="LoRA Inference")
baseline_df['lora_text'] = baseline_df['complex_text'].progress_apply(generate_with_lora)


LoRA Inference:   0%|          | 0/72 [00:00<?, ?it/s]

In [25]:
# Берем случайный пример из тестовой выборки
sample = baseline_df.sample(1).iloc[0]

print("=== СРАВНЕНИЕ ===\n")

print("Оригинальный академический текст):")
print(sample['complex_text'])
print("-" * 50)

print("Псевдотаргет Llama-3-8B):")
print(sample['simplified_text'])
print("-" * 50)

print("Бейзлайн (FLAN-T5):")
print(sample['baseline_text'])
print("-" * 50)

print("После обучения (FLAN-T5 + LoRA):")
print(sample['lora_text'])

=== СРАВНЕНИЕ ===

Оригинальный академический текст):
In this article, we examine what factors influence scope interpretation in Russian, a case-marking, free-word-order language. We focus on the Russian equivalents of the English simple transitive sentences in (1a) and (2a), with quantifiers in subject and object positions.
--------------------------------------------------
Псевдотаргет Llama-3-8B):
What makes it hard to understand Russian sentences? 

In this article, we look at what makes it tricky to figure out what's happening in a Russian sentence. We're going to focus on simple sentences in English that have a doer and something that gets done, like "John kicks the ball".
--------------------------------------------------
Бейзлайн (FLAN-T5):
The Russian equivalents of the English simple transitive sentences in (1a) and (2a) are based on a case-marking, free-word-order language.
--------------------------------------------------
После обучения (FLAN-T5 + LoRA):
The Russian equiva

In [26]:

# Сохраняем финальную таблицу со всеми сравнениями
final_results_path = '/content/drive/MyDrive/AppliedNN_papers/final_comparison.csv'
baseline_df.to_csv(final_results_path, index=False)
print(f"Финальные результаты сохранены в: {final_results_path}")

Финальные результаты сохранены в: /content/drive/MyDrive/AppliedNN_papers/final_comparison.csv


### Оценка результатов

In [28]:
# 1. Расчет ROUGE (Сравнение LoRA-генерации с эталоном Llama-3)
rouge = evaluate.load("rouge")

rouge_results = rouge.compute(
    predictions=baseline_df['lora_text'].tolist(),
    references=baseline_df['simplified_text'].tolist()
)


In [29]:
# 2. Расчет косинусного сходства (Сохранение смысла оригинала)
# all-MiniLM-L6-v2 — легкая и быстрая модель для Sentence-to-Sentence задач
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Загрузка модели для построения эмбеддингов...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")

ROUGE-1: 0.1620
ROUGE-2: 0.0651
ROUGE-L: 0.1392


In [33]:
#Вычисление эмбеддингов и косинусного сходства
# Получаем векторы для оригинальных академических текстов и наших упрощений
embeddings_original = embedding_model.encode(baseline_df['complex_text'].tolist(), convert_to_tensor=True)
embeddings_simplified = embedding_model.encode(baseline_df['lora_text'].tolist(), convert_to_tensor=True)


In [34]:
# Считаем косинусное сходство попарно
cosine_scores = util.cos_sim(embeddings_original, embeddings_simplified).diagonal()


In [36]:
# Добавляем результаты в датафрейм для последующего анализа
baseline_df['cosine_similarity'] = cosine_scores.cpu().numpy()


In [37]:
mean_cosine = baseline_df['cosine_similarity'].mean()

print(f"Среднее косинусное сходство: {mean_cosine:.4f}")


Среднее косинусное сходство: 0.4419


In [38]:
# Сохраняем обновленный датафрейм с метриками
baseline_df.to_csv('/content/drive/MyDrive/AppliedNN_papers/final_metrics_evaluation.csv', index=False)

In [40]:
# Сортируем датафрейм по возрастанию косинусного сходства
errors_df = baseline_df.sort_values(by='cosine_similarity', ascending=True).head(5)

print("=== АНАЛИЗ ОШИБОК: ТОП ХУДШИХ УПРОЩЕНИЙ ПО СОХРАНЕНИЮ СМЫСЛА ===\n")

for index, row in errors_df.iterrows():
    print(f"Косинусное сходство: {row['cosine_similarity']:.4f}")
    print(f"ОРИГИНАЛ: {row['complex_text']}")
    print("-" * 40)
    print(f"УПРОЩЕНИЕ (LoRA): {row['lora_text']}")
    print("=" * 80 + "\n")

=== АНАЛИЗ ОШИБОК: ТОП ХУДШИХ УПРОЩЕНИЙ ПО СОХРАНЕНИЮ СМЫСЛА ===

Косинусное сходство: -0.0606
ОРИГИНАЛ: 10See a. o. Ward and Hirschberg ; Pierrehumbert and Steele ; Hirschberg and Ward ; examples from Bolinger and Jackendoﬀ. 11From Ladd . I notate the ﬁnal R at the end of sentence here, indicating that it is not part of the pitch accent on the prominent word— all in ; see Hara , Oshima and Tomioka for Japanese 13O’Connor and Arnold , Ladd , examples (16) and (19).
----------------------------------------
УПРОЩЕНИЕ (LoRA): rewrite for a 10-12 year old

Косинусное сходство: -0.0404
ОРИГИНАЛ: that the referential reading is by no means the only one available to odin/one indefinites; it is just that it is chosen more often for indefinites of this type than for others. Conversely, the referential or choice-function reading is in principle available to dva/two indefinites (which is why they are able to scope out of islands), but the quantificational reading is accessed more readily.
-------

**Чувствительность к неформатному тексту (Сноски и цитирования):** Модель демонстрирует критическое падение качества (галлюцинации), когда на вход подаются фрагменты библиографии, списки авторов или сноски. В таких случаях, не находя семантического ядра для упрощения, модель-ученик вместо перефразирования возвращает системный префикс задачи (prompt leakage). Это указывает на необходимость более строгой эвристической очистки датасета от ссылочного аппарата перед инференсом.

In [41]:
# Ищем все случаи, где модель выдала кусок промпта (по ключевым словам)
leakage_mask = baseline_df['lora_text'].str.contains('10-12 year', case=False, na=False)
leakage_count = leakage_mask.sum()

print(f"Найдено случаев 'утечки промпта' (галлюцинаций на мусорных данных): {leakage_count}")

Найдено случаев 'утечки промпта' (галлюцинаций на мусорных данных): 23


In [42]:

# Отсеиваем эти строки, создавая очищенный датафрейм
clean_df = baseline_df[~leakage_mask].copy()

# Пересчитываем среднее косинусное сходство без учета мусора
clean_mean_cosine = clean_df['cosine_similarity'].mean()
print(f"Новое среднее косинусное сходство (чистое): {clean_mean_cosine:.4f}\n")

Новое среднее косинусное сходство (чистое): 0.5972



In [44]:
len(clean_df)

49

In [45]:
# Ищем новые топ-3 худших примеров на очищенных данных
errors_df_clean = clean_df.sort_values(by='cosine_similarity', ascending=True).head(10)

print("=== АНАЛИЗ ОШИБОК (ОЧИЩЕННАЯ ВЫБОРКА) ===\n")

for index, row in errors_df_clean.iterrows():
    print(f"Косинусное сходство: {row['cosine_similarity']:.4f}")
    print(f"ОРИГИНАЛ: {row['complex_text']}")
    print("-" * 40)
    print(f"УПРОЩЕНИЕ (LoRA): {row['lora_text']}")
    print("=" * 80 + "\n")

=== АНАЛИЗ ОШИБОК (ОЧИЩЕННАЯ ВЫБОРКА) ===

Косинусное сходство: 0.0737
ОРИГИНАЛ: 1. Some discussion of the relationship between head shakes and so-called ‘N-gestures’ which are manual gestures using an open hand with the palm oriented away from the speaker and which are often used in expressions of negation will be found in Kendon (Forthcoming).
----------------------------------------
УПРОЩЕНИЕ (LoRA): 1.

Косинусное сходство: 0.1008
ОРИГИНАЛ: 5.1. THE STATUS OF THE PRINCIPLE. Maintaining that the RP is a universal principle means that individual languages or dialects cannot differ with respect to whether they obey it; crucially ambiguous factors of the appropriate sort would always resolve feature conflicts, and (in the absence of principled res- olution) factors not satisfying the conditions in 43 would always fail to resolve
----------------------------------------
УПРОЩЕНИЕ (LoRA): a conflict.

Косинусное сходство: 0.1487
ОРИГИНАЛ: In section 1 above I suggested that the answer in

In [47]:
# Ищем новые топ-3 худших примеров на очищенных данных
errors_df_clean = clean_df.sort_values(by='cosine_similarity', ascending=False).head(10)

print("=== Лучшие результаты ===\n")

for index, row in errors_df_clean.iterrows():
    print(f"Косинусное сходство: {row['cosine_similarity']:.4f}")
    print(f"ОРИГИНАЛ: {row['complex_text']}")
    print("-" * 40)
    print(f"УПРОЩЕНИЕ (LoRA): {row['lora_text']}")
    print("=" * 80 + "\n")

=== Лучшие результаты ===

Косинусное сходство: 1.0000
ПСЕВДО-ТАРГЕТ: In Russian, some sentences have the object first. To make it easier to read, we translate these sentences into English using the passive voice. This means the object comes first, but we note that the original Russian sentence is in the active voice.
----------------------------------------
УПРОЩЕНИЕ (LoRA): In Russian, both . For readability, we use the passive voice to translate the OVS sentences into English, in order to indicate that the object comes first, but we note that the corresponding Russian sentences are in the active voice.

Косинусное сходство: 0.9963
ПСЕВДО-ТАРГЕТ: The article is divided into four parts. First, we'll talk about how words are mixed up in Russian, what they mean, and how they work. Then, we'll share the results of our experiment. After that, we'll explain what we found and suggest what others might want to study next.
----------------------------------------
УПРОЩЕНИЕ (LoRA): The article

## Выводы по результатам оценки качества генерации

На основе рассчитанных метрик и качественного анализа текстов можно сделать следующие выводы:

**1. Количественные показатели**
Метрики ROUGE (ROUGE-1: 0.1620, ROUGE-L: 0.1392) и среднее косинусное сходство (0.4419) демонстрируют умеренный уровень совпадения. Для задачи глубокого перефразирования научных текстов это ожидаемый результат, так как упрощение предполагает значительное изменение структуры предложений и замену лексики, что снижает прямое пословное совпадение (n-граммы).

**2. Проблема чистоты данных (Prompt Leakage)**
Выявлено 23 случая генерации системного префикса вместо ответа. Анализ показал, что это происходит исключительно на шумовых данных: библиографических ссылках, номерах сносок и технических фрагментах. Не находя в них семантического ядра для упрощения, модель возвращает текст промпта. Это указывает на необходимость более строгой эвристической очистки датасета.

**3. Анализ ошибок на очищенных данных**
Худшие результаты демонстрируют тенденцию модели к чрезмерному сокращению. В сложных случаях модель отбрасывает зависимые предикативные части или теряет контекст, генерируя обрывочные фрагменты: единичные имена авторов, номера пунктов или отдельные слова. Это свидетельствует о сложностях с обработкой сильно фрагментированных абзацев.

**4. Анализ успешных генераций**
В случаях с высоким косинусным сходством (0.85–1.0) модель успешно справляется с поставленной задачей. Наблюдаются следующие паттерны:

* Корректная замена сложного академического синтаксиса на более простые конструкции.
* Грамотное разбиение длинных предложений на несколько коротких.
* Сохранение ключевой фактологии и терминологии при общем снижении сложности текста (например, разъяснение использования пассивного залога при переводе).